# v1: InceptionV3 + LSTM — Baseline Image Captioning

| | |
|---|---|
| **Framework** | TensorFlow / Keras |
| **Dataset** | Instagram Images with Captions (`prithvijaunjale/instagram-images-with-captions`) |
| **Image Encoder** | InceptionV3 — second-to-last layer (`layers[-2]`), flat 2048-dim vector |
| **Text Decoder** | Keras Embedding + LSTM(256) |
| **BLEU-4** | Not formally measured (no ground-truth references per image) |
| **Status** | Baseline — superseded by v2 (attention) |

---

## What This Notebook Does

This is the first working image captioning model in this project. It establishes the baseline pipeline:
load Instagram images → extract InceptionV3 features → clean captions → tokenize →
train an LSTM decoder → generate captions at inference time.

Two model variants are explored:
- **`model`**: combines image and text features via `Add()` (element-wise sum)
- **`model1`**: combines via `Concatenate()` with L2 regularization

Both were trained on the same data. The goal was to establish which combination strategy works better
before moving on to more complex attention mechanisms in v2.

**What we learn from v1:** The architecture is sound, but the captions are wrong —
the model collapses into generic phrases like 'I love my new' regardless of what the image shows.
The problem is not the architecture — it is the data. Instagram captions are emotional and promotional,
not visual descriptions. This diagnosis drives every decision made in v2 and v3.

## Tech Stack

In [ ]:
# Core framework
import tensorflow as tf          # 2.19.0 — training, data pipelines, model saving
import numpy as np               # 1.24.x — array operations
import pandas as pd              # 1.5.x  — reading CSV files

# Keras components used in this notebook
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.layers import (
    Input, Dense, Embedding, LSTM, Add, Concatenate, Dropout
)
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, Callback
)
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Utilities
import os
import re
import pickle
import random
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import concurrent.futures

# BLEU evaluation
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

---

## 1. Dataset: Instagram Images with Captions

**Source:** Kaggle — `prithvijaunjale/instagram-images-with-captions`

The dataset contains two folders:
- `instagram_data/` — first batch (~17k images)
- `instagram_data2/` — second batch (~18k images)

Each folder has a CSV file pairing image filenames with their captions. Total raw dataset: ~34,926 image-caption pairs.

**Why Instagram data?**
The original goal was to generate Instagram-style captions — casual, expressive, sometimes hashtagged.
Existing captioning models (trained on Flickr30k, COCO) generate dry, literal descriptions
like 'a dog sitting on a bench'. Instagram-style would be 'Weekend vibes 🐾'.
This seemed like an interesting differentiation.

**Why it ultimately failed:**
Instagram captions are not visually grounded. A caption like 'Link in bio!' or 'I love my new collab!'
has nothing to do with what is in the image. The model cannot learn any correspondence between
image features and caption tokens — there is none. Instead it learns the prior distribution of
captions: common phrases that appear frequently regardless of image content.
This is why every model in this notebook generates 'I love my new...' for every input image.
The fix (switching to Flickr30k + COCO) does not happen until v3.

In [ ]:
# Dataset paths (Kaggle notebook environment)
# Each entry is (image_folder, caption_csv)
data_folders = [
    (
        '/kaggle/input/instagram-images-with-captions/instagram_data/img',
        '/kaggle/input/instagram-images-with-captions/instagram_data/captions_csv.csv'
    ),
    (
        '/kaggle/input/instagram-images-with-captions/instagram_data2/img2',
        '/kaggle/input/instagram-images-with-captions/instagram_data2/captions_csv2.csv'
    )
]

### 1.1 Loading Captions

The two CSV files have slightly different column structures:
- `captions_csv.csv` has columns `['Image File', 'Caption']`
- `captions_csv2.csv` has no header — columns are inferred from first row

The loader handles both formats and builds a dict: `{image_path: raw_caption_string}`.
Only images that actually exist on disk are included (some paths in the CSV
do not correspond to real files).

In [ ]:
def load_captions(data_folders):
    """Load image-caption pairs from one or more CSV-backed folders.

    Returns a dict mapping absolute image paths to their raw caption strings.
    Images that do not exist on disk are silently skipped.
    """
    with_caption = {}
    for img_folder, caption_file in data_folders:
        df = pd.read_csv(caption_file)
        df.columns = df.columns.str.strip()
        print(f'Folder: {img_folder}\n  Columns: {df.columns.tolist()}')

        # Handle two different CSV layouts
        if 'Image File' in df.columns and 'Caption' in df.columns:
            rows = df[['Image File', 'Caption']].values
        else:
            # captions_csv2 has no header — assign column names
            df = df.set_axis(['Sr No', 'Image File', 'Caption'], axis=1)
            rows = df[['Image File', 'Caption']].values

        for img_file, caption in rows:
            img_path = os.path.join(os.path.dirname(img_folder), str(img_file) + '.jpg')
            if os.path.exists(img_path):
                with_caption[img_path] = caption
            # else: silently skip — CSV has phantom entries

    return with_caption


captions_dict = load_captions(data_folders)
print(f'\nTotal image-caption pairs loaded: {len(captions_dict)}')

---

## 2. Caption Cleaning

Instagram captions contain noise that would harm a language model:
- **URLs** (`https://bit.ly/...`, `www.shop.com`) — not language, just noise
- **@Mentions and #Hashtags** — vary per account, models cannot learn them meaningfully
- **Emojis** — Unicode characters outside the ASCII range; need careful handling
  because multiple emojis in sequence are a single character run in Unicode,
  not separate tokens
- **Skin tone modifiers** — emoji modifiers like 🏻🏿 that attach to base emojis
  and cause tokenization artifacts
- **Extra whitespace** — artifacts of the above removals

Each caption also gets `<start>` and `<end>` tokens wrapping the text.
These special tokens teach the model where a caption begins and ends,
which is essential for the autoregressive inference loop.

**Why `<start>` specifically:**
At inference time, we feed `<start>` as the seed token and let the model predict
the next word. Without a designated start token, the model has no signal
about where to begin.

**Why `<end>` specifically:**
The model needs to learn to stop. Without `<end>`, the greedy decode loop
runs forever (or until a max_len cutoff). Training with `<end>` teaches
the model to assign high probability to this token when the caption is complete.

In [ ]:
def clean_caption(caption):
    """Clean a raw Instagram caption and wrap it with <start> / <end> tokens.

    Operations applied in order:
    1. Remove URLs (http/https/www)
    2. Remove @mentions and #hashtags
    3. Separate Unicode supplementary plane chars (catches most emojis)
    4. Separate emojis in the standard Unicode emoji range (U+1F300–U+1FAFF)
    5. Remove skin tone modifiers (U+1F3FB–U+1F3FF) to prevent tokenization
       artifacts where 'person+dark_skin' becomes an untokenizable merged char
    6. Collapse repeated whitespace
    7. Wrap with <start> and <end>

    Returns a fallback '<start> <end>' string for non-string inputs (NaN rows).
    """
    if not isinstance(caption, str):
        return '<start> <end>'

    # Step 1: Remove URLs
    caption = re.sub(r'https?://\S+|www\.\S+', '', caption).strip()

    # Step 2: Remove mentions and hashtags
    caption = re.sub(r'[@#]\w+', '', caption).strip()

    # Step 3: Separate supplementary Unicode plane characters (U+10000 and above)
    # Without this, a sequence like '🎉🎊' is one char run and becomes one token
    caption = re.sub(r'([\U00010000-\U0010ffff])', r' \1 ', caption)

    # Step 4: Separate emojis in standard emoji ranges
    caption = re.sub(r'([\U0001F300-\U0001FAFF\u2600-\u27BF])', r' \1 ', caption)

    # Step 5: Remove skin tone modifier codepoints
    caption = re.sub(r'[\U0001F3FB-\U0001F3FF]', '', caption)

    # Step 6: Collapse whitespace
    caption = re.sub(r'\s+', ' ', caption).strip()

    # Step 7: Add special tokens
    return '<start> ' + caption + ' <end>'


# Apply cleaning to all captions and store in a new dict
updated_captions_dict = {
    k: clean_caption(v)
    for k, v in captions_dict.items()
}

print(f'Original: {len(captions_dict)} pairs')
print(f'After cleaning: {len(updated_captions_dict)} pairs')

# Sample a few to verify
for k, v in list(updated_captions_dict.items())[:3]:
    print(f'  {v[:80]}')

---

## 3. Tokenization with Keras TextVectorization

### What is tokenization?
Tokenization converts text into integers — the only thing a neural network can process.
Each unique word (or subword, depending on the tokenizer) gets a unique integer ID.
A caption like `'<start> a dog runs <end>'` becomes `[2, 14, 391, 876, 3]`.

### Why Keras TextVectorization?
At this point (v1), the project uses TF/Keras throughout, and `TextVectorization`
is the native Keras tokenizer. It integrates cleanly with `tf.data` pipelines
and can be saved inside the model for deployment.

In v3, this is replaced by the **GPT-2 tokenizer** (BPE-based), which operates
at the subword level and handles out-of-vocabulary words better.

### Vocabulary size choice: 10,000 tokens
The full unconstrained vocabulary from the Instagram dataset is ~33,647 unique tokens.
But most of these appear only once or twice — they are hashtags, brand names,
misspellings, or emoji combinations that the model cannot learn meaningfully
from a single occurrence.

A frequency analysis showed that the top ~16,000 tokens cover 93% of all word occurrences.
Setting `max_tokens=10000` cuts the tail aggressively:
- Reduces the softmax output layer size from ~33k to 10k — faster training
- Forces the model to focus on common, learnable words
- Rare words get mapped to `[UNK]` (index 1), which is acceptable for captioning

**Later lesson (v3):** Even 10k is too large for a ~21k Instagram dataset.
The GPT-2 tokenizer on Flickr30k/COCO has a fixed 50,257-token BPE vocabulary
but is pre-trained — it already knows how to represent any English text,
so there is no cold-start problem.

### What `standardize=None` means
When `standardize=None` is set, TextVectorization does not touch the text at all — no lowercasing, no punctuation stripping. This is intentional for Instagram captions because capitalization carries meaning in this context (e.g., 'I LOVE this' vs 'i love this' express different intensities). Instagram captions are informal and expressive — standardizing them would destroy signal that the model could potentially learn from.

In [ ]:
# Build vocabulary from the cleaned caption list
caption_list = list(updated_captions_dict.values())

vectorizer = TextVectorization(
    max_tokens=10000,       # Keep only top 10k most frequent tokens
    standardize=None,       # No additional lowercasing/punctuation stripping
    split='whitespace',     # Split on spaces — we pre-tokenized above
    output_mode='int',      # Return integer IDs, not one-hot or TF-IDF
    output_sequence_length=None,  # Variable length — we pad manually later
)

# .adapt() scans the full corpus and builds the vocabulary index
# Token 0 = padding/mask, Token 1 = [UNK] (out-of-vocabulary)
# Token 2 onward = actual words sorted by frequency (most frequent first)
vectorizer.adapt(caption_list)

# Vectorize all captions
x = vectorizer(caption_list)

vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)

# max_len: longest caption in the dataset (in tokens)
# Used for padding sequences to a fixed length
max_len = x.shape[1]

print(f'Vocabulary size: {vocab_size}')
print(f'Max caption length (tokens): {max_len}')
print(f'First 10 vocab entries: {vocab[:10]}')
print(f'Sample tokenized caption: {x[0].numpy()[:15]}')

---

## 4. Image Feature Extraction — InceptionV3

### Why InceptionV3?
InceptionV3 is a well-established CNN image classifier pretrained on ImageNet (1000 classes).
Its deep convolutional layers learn to extract increasingly abstract visual features:
early layers detect edges and textures; later layers detect objects and scenes.
We want the rich semantic representation from the final layers.

### Why `layers[-2]` (second-to-last layer)?
The very last layer (`layers[-1]`) is the classification head — a softmax over 1000 ImageNet classes.
This is useless for captioning because:
1. It collapses all information into a probability distribution over 1000 categories
2. We need a continuous feature vector, not class probabilities

The second-to-last layer (`layers[-2]`) is the Global Average Pooling layer
that produces a flat **2048-dimensional vector** representing the entire image.
This is a high-level semantic embedding — the most information-rich
single vector we can extract from InceptionV3.

**Output shape:** `(1, 2048)` per image → squeezed to `(2048,)` when stored.

### Why not fine-tune InceptionV3?
InceptionV3 is used as a frozen feature extractor. The classification head is removed and features are extracted from `layers[-2]`, giving a 2048-dimensional representation per image.

**v2 upgrade:** v2 switches from `layers[-2]` (flat 2048) to `mixed10` (spatial 64×2048),
which preserves spatial information and enables attention over image regions.

In [ ]:
# Build the feature extractor from InceptionV3
# Input: 299×299 RGB image
# Output: 2048-dim feature vector (flat)

base_model = InceptionV3(weights='imagenet')  # Full classification model
new_input = base_model.input
hidden_layer = base_model.layers[-2].output   # Global Average Pooling output: (2048,)

image_feature_extract_model = Model(
    inputs=new_input,
    outputs=hidden_layer
)
# All InceptionV3 weights are frozen — we do not train them
image_feature_extract_model.trainable = False

print(f'Feature extractor output shape: {image_feature_extract_model.output_shape}')

In [ ]:
def preprocess(img_path, target_size=(299, 299)):
    """Load and preprocess an image for InceptionV3.

    InceptionV3 expects pixel values in [-1, 1] (from preprocess_input),
    not the [0, 1] or [0, 255] range used by other models.
    """
    img = keras_image.load_img(img_path, target_size=target_size)
    img = keras_image.img_to_array(img)          # (299, 299, 3), values [0, 255]
    img = np.expand_dims(img, axis=0)            # (1, 299, 299, 3)
    img = preprocess_input(img)                  # Rescales to [-1, 1] for InceptionV3
    return img


def extract_features_batch(img_paths, model, batch_size=32):
    """Extract InceptionV3 features for a list of image paths.

    Processes in batches for efficiency. Returns dict {path: feature_array}.
    Features are stored as float16 to halve memory usage (2048-dim × 4 bytes → × 2 bytes).
    """
    features = {}
    for i in tqdm(range(0, len(img_paths), batch_size)):
        batch_paths = img_paths[i:i + batch_size]
        batch_imgs = np.stack([preprocess(p)[0] for p in batch_paths], axis=0)
        batch_features = model.predict(batch_imgs, verbose=0).astype('float16')
        for path, feat in zip(batch_paths, batch_features):
            features[path] = feat
    return features


# ── Feature extraction was run once and saved. Commented out to avoid re-running.
# ── Re-running would take ~2 hours on a T4 GPU for 21k images.
# ── To reproduce: uncomment and run (requires dataset on disk)

# image_features = extract_features_batch(
#     list(updated_captions_dict.keys()),
#     image_feature_extract_model
# )
# pickle.dump(image_features, open('image_features.pkl', 'wb'))

# ── Load pre-computed features from disk
image_features_dict = pickle.load(
    open('/kaggle/input/datasets/huzefamerchant/image-features-attention/image_features.pkl', 'rb')
)
print(f'Features loaded: {len(image_features_dict)} images')
print(f'Feature shape per image: {list(image_features_dict.values())[0].shape}')

---

## 5. Train / Validation Split

90% of images go to training, 10% to validation. `random_state=42` ensures
reproducibility — the same split is used across all v1 training runs.

**One caption per image:** Instagram data has exactly one caption per image.
This is a fundamental limitation compared to Flickr30k and COCO (5 captions/image).
With multiple captions, you can train on all of them and evaluate on the others.
With one caption, the model gets a single noisy training signal per image,
which makes it harder to learn generalizable visual-linguistic associations.

In [ ]:
# Only use images that have both a caption AND pre-computed features
image_keys = [
    k for k in updated_captions_dict.keys()
    if k in image_features_dict
]

train_keys, val_keys = train_test_split(
    image_keys,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(f'Total usable images: {len(image_keys)}')
print(f'Train: {len(train_keys)} | Val: {len(val_keys)}')

---

## 6. Data Preparation — Sequence Construction

### How teacher forcing works
Image captioning is trained with **teacher forcing**: at each timestep,
the decoder receives the *ground-truth* previous word, not its own previous prediction.

For a caption `[<start>, 'a', 'dog', 'runs', <end>]` (vectorized to `[2, 14, 391, 876, 3]`):
- **Input sequence (x_seq):** `[2, 14, 391, 876]` — caption without the last token
- **Target sequence (y):** `[14, 391, 876, 3]` — caption without the first token

At each position `t`, the model sees tokens `0..t` and must predict token `t+1`.
This is equivalent to maximum likelihood estimation: maximize P(next_word | image, previous_words).

### Why pre-compute x_seq and y?
These arrays are constant across epochs. Computing them inside the data generator
every time would be wasteful. Pre-computing and storing in dicts
(`x_all_timesteps_dict`, `y_all_timesteps_dict`) means the generator only
does array lookups during training.

### Padding strategy
All sequences are padded to `max_len` tokens with zeros (`value=0`).
The model learns that 0 = padding and ignores those positions.
Post-padding is used (zeros appended at the end) so the meaningful
tokens stay left-aligned and the LSTM processes them in natural order.

In [ ]:
# Pre-compute input/target sequences for all captions
# x_seq = caption[:-1] (all tokens except last)
# y      = caption[1:]  (all tokens except first — shifted by one)

x_all_timesteps_dict = {}
y_all_timesteps_dict = {}

for key, caption in tqdm(updated_captions_dict.items()):
    # Vectorize caption → integer sequence
    z = vectorizer([caption])[0].numpy().astype('int32')

    # Input: caption without <end> token, padded to max_len
    x_all_timesteps_dict[key] = pad_sequences(
        [z[:-1]], maxlen=max_len, padding='post', value=0
    )[0]

    # Target: caption without <start> token, padded to max_len
    y_all_timesteps_dict[key] = pad_sequences(
        [z[1:]], maxlen=max_len, padding='post', value=0
    )[0]

print('Sequence construction done.')
# Sanity check — show one example
sample_key = train_keys[0]
sample_cap = updated_captions_dict[sample_key]
print(f'Caption: {sample_cap[:60]}...')
print(f'x_seq[:8]: {x_all_timesteps_dict[sample_key][:8]}')
print(f'y[:8]:     {y_all_timesteps_dict[sample_key][:8]}')

### 6.1 Batch Generator

A **custom generator** is used instead of loading all data into memory at once.
The full feature + sequence dataset is ~2GB — too large for a Kaggle T4 notebook.

The generator:
1. Shuffles the keys at the start of each pass (so each epoch sees a different order)
2. Iterates through keys, appending to batch lists
3. Yields a complete batch of `(image_features, input_sequences, target_sequences)`
   once `batch_size` items are accumulated
4. Runs in a `while True` loop — `tf.data` calls it as needed

**Why not use a Keras `Sequence` or pure `tf.data`?**
The image features are stored as a Python dict (loaded from pickle).
Dict lookups are fast and fit naturally into a generator pattern.
A `tf.data.Dataset.from_generator` wraps it into the TF data pipeline
with prefetching for better GPU utilization.

In [ ]:
BATCH_SIZE = 64


def batch_generator(keys, image_features_dict, x_dict, y_dict,
                    max_len=max_len, batch_size=BATCH_SIZE):
    """Infinite generator yielding ((image_batch, seq_batch), target_batch).

    Keys are shuffled each pass so the model sees a different sample order
    every epoch, preventing it from memorizing ordering artifacts.
    """
    while True:
        x_image_batch, x_seq_batch, y_batch = [], [], []
        np.random.shuffle(keys)  # In-place shuffle each epoch

        for key in keys:
            # Image features: cast to float32 (stored as float16 to save memory)
            image_feature = tf.cast(image_features_dict[key], tf.float32)
            x_image_batch.append(image_feature)
            x_seq_batch.append(x_dict[key])
            y_batch.append(y_dict[key])

            if len(y_batch) == batch_size:
                yield (
                    (np.array(x_image_batch), np.array(x_seq_batch)),
                    np.array(y_batch)
                )
                x_image_batch, x_seq_batch, y_batch = [], [], []


# Wrap in tf.data for prefetching (loads next batch while GPU trains on current)
train_dataset = tf.data.Dataset.from_generator(
    lambda: batch_generator(
        train_keys, image_features_dict,
        x_all_timesteps_dict, y_all_timesteps_dict
    ),
    output_signature=(
        (
            tf.TensorSpec((BATCH_SIZE, 2048), tf.float32),
            tf.TensorSpec((BATCH_SIZE, max_len), tf.int32)
        ),
        tf.TensorSpec((BATCH_SIZE, max_len), tf.int32)
    )
).repeat().prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: batch_generator(
        val_keys, image_features_dict,
        x_all_timesteps_dict, y_all_timesteps_dict
    ),
    output_signature=(
        (
            tf.TensorSpec((BATCH_SIZE, 2048), tf.float32),
            tf.TensorSpec((BATCH_SIZE, max_len), tf.int32)
        ),
        tf.TensorSpec((BATCH_SIZE, max_len), tf.int32)
    )
).repeat().prefetch(tf.data.AUTOTUNE)

train_steps = len(train_keys) // BATCH_SIZE
val_steps = len(val_keys) // BATCH_SIZE

print(f'Train steps/epoch: {train_steps} | Val steps/epoch: {val_steps}')

---

## 7. Model Architecture

### Overview
The model has two input branches that are combined into a single output:

```
Image → Dense(512) → Dense(512) → Dropout → ┐
                                              ├─ Add() or Concat() → Dense(vocab)
Caption → Embedding → LSTM(512) → Dropout → ┘
```

**Image branch:**
The flat 2048-dim InceptionV3 feature is passed through two Dense layers
to project it into the same dimensionality as the LSTM output.
Without this projection, combining 2048-dim image features with 256-dim text
features would create an imbalanced input to the fusion layer.

**Text branch:**
An Embedding layer maps token IDs to 256-dim dense vectors.
The LSTM processes the sequence and outputs a single context vector
summarizing all previous tokens. This is the decoder's representation
of 'what has been said so far'.

**`mask_zero=True` in Embedding:**
Tells Keras that token ID 0 is padding and should not contribute to
the LSTM's computations. The LSTM ignores zero-padded timesteps.
Without this, padding tokens would corrupt the LSTM hidden state.

### Two variants: Add vs Concatenate

**`model` — Add:**
Element-wise sum of image and text features. Requires both to have the
same dimension (achieved by projecting image to 256-dim to match LSTM output).
Addition creates a tight coupling: each image feature dimension directly
modifies the corresponding text dimension. Simple and parameter-efficient.

**`model1` — Concatenate + L2:**
Concatenates 512-dim image features with 512-dim LSTM output → 1024-dim combined vector.
A Dense(256) then reduces this. Concatenation preserves all information
from both branches independently — the model learns which features matter.
L2 regularization was added because the larger model overfitted faster.

**Hypothesis behind trying both:**
It was unclear whether the image and text representations should be additively combined
(forcing them into a shared space) or concatenated (letting the model learn the fusion).
In practice, both performed similarly on Instagram data because the fundamental problem
(data quality) dominated over architecture choice.

In [ ]:
# ─── Model variant 1: Add() fusion ───────────────────────────────────────────

# Image branch: project 2048 → 512 → 512 → dropout
image_input = Input(shape=(2048,), name='image_input')
image_dense1 = Dense(512, activation='relu')(image_input)
image_dense2 = Dense(512, activation='relu')(image_dense1)
image_features_proj = Dropout(0.4)(image_dense2)

# Text branch: embed → LSTM(512) → dropout
caption_input = Input(shape=(max_len,), name='caption_input')
embeddings = Embedding(
    input_dim=vocab_size, output_dim=256, mask_zero=True
)(caption_input)
lstm_out = LSTM(512)(embeddings)  # Returns single vector: (batch, 512)
lstm_out = Dropout(0.3)(lstm_out)

# ── Note: image_features_proj is 512-dim, lstm_out is 512-dim
# ── Add() requires matching dimensions — that's why the image is projected to 512
merged = Add()([image_features_proj, lstm_out])
merged = Dense(256, activation='relu')(merged)
decoder_output = Dense(vocab_size, activation='softmax')(merged)

model = Model(inputs=[image_input, caption_input], outputs=decoder_output)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)
model.summary()

In [ ]:
# ─── Model variant 2: Concatenate() fusion + L2 regularization ───────────────
# Added L2 regularization because model1's higher capacity (concat = 1024-dim)
# caused faster overfitting on the small Instagram dataset.
# kernel_regularizer=l2(1e-4) penalizes large weights, encouraging simpler solutions.

image_input_1 = Input(shape=(2048,), name='image_input')
image_d1 = Dense(512, activation='relu')(image_input_1)
# ── 256 here (not 512) because Concat will combine with 512-dim LSTM output
# ── giving 256+512=768, then Dense(256) reduces it
image_d2 = Dense(256, activation='relu')(image_d1)
image_proj = Dropout(0.4)(image_d2)

caption_input_1 = Input(shape=(max_len,), name='caption_input')
emb_1 = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True)(caption_input_1)
# L2 on both kernel (input weights) and recurrent kernel (hidden state weights)
lstm_1 = LSTM(
    512,
    kernel_regularizer=l2(1e-4),
    recurrent_regularizer=l2(1e-4)
)(emb_1)
lstm_1 = Dropout(0.3)(lstm_1)

# Concat: 256-dim image + 512-dim text = 768-dim
merged_1 = Concatenate()([image_proj, lstm_1])
merged_1 = Dense(256, activation='relu')(merged_1)
merged_1 = Dropout(0.3)(merged_1)
output_1 = Dense(vocab_size, activation='softmax')(merged_1)

model1 = Model(inputs=[image_input_1, caption_input_1], outputs=output_1)
model1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)
model1.summary()

---

## 8. Training

### Callbacks used

**ModelCheckpoint** — saves the model whenever validation loss improves.
Only the best model is kept (`save_best_only=True`).
This is important because early stopping restores to the best epoch —
if the checkpoint is not saved, the best weights are lost.

**ReduceLROnPlateau** — if validation loss has not improved for `patience` epochs,
the learning rate is multiplied by `factor` (reduced).
Lower LR allows the optimizer to make finer updates near the optimum.
Here: factor=0.9 (gentle reduction), patience=2, min_lr=1e-5.

**EarlyStopping** — stops training if val_loss has not improved for `patience` epochs.
Prevents wasting compute and overfitting. `restore_best_weights=True` loads
the best checkpoint at the end of training.

**Note on v2 improvement:** v2 replaces `ReduceLROnPlateau` with `ReduceLRReloadBest`
— a custom callback that also reloads the best model weights whenever the LR is reduced.
Standard `ReduceLROnPlateau` only reduces the LR but continues from the current
(potentially worse) weights. `ReduceLRReloadBest` combines LR reduction with
weight restoration, avoiding the "reduced LR on bad weights" problem.

In [ ]:
# Training callbacks
checkpoint_callback = ModelCheckpoint(
    '/kaggle/working/best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

reduce_lr_callback = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.9,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
# ── Training was run on Kaggle (T4 GPU). Commented out here for inference-only use.
# ── The model ran for 50 epochs max with early stopping.
# ── To reproduce training: uncomment, ensure data is loaded, run on GPU.

# history = model.fit(
#     train_dataset,
#     epochs=50,
#     steps_per_epoch=train_steps,
#     validation_data=val_dataset,
#     validation_steps=val_steps,
#     verbose=1,
#     callbacks=[
#         checkpoint_callback,
#         reduce_lr_callback,
#         early_stopping_callback
#     ]
# )

# Load the best saved model for inference
# model = tf.keras.models.load_model('/kaggle/input/.../best_model.keras')

print('Training section — run on Kaggle.')
print('See Kaggle notebook: https://www.kaggle.com/huzefamerchant/image-captioning-1')

---

## 9. Inference — Greedy Decoding

**Greedy decoding** generates captions one token at a time.
At each step, the model produces a probability distribution over the vocabulary
and the token with the highest probability is selected.

### Algorithm
```
1. Start with seed sequence: [<start>]
2. Feed (image_feature, current_sequence) → model → probability distribution
3. Pick argmax (highest probability token)
4. Append that token to the sequence
5. Repeat until <end> token is produced or max_len is reached
6. Decode the integer sequence back to words using the vocabulary
```

### Why greedy is the simplest but weakest choice
Greedy decoding is deterministic and fast. But it can make locally optimal
choices that lead to globally poor captions: once a wrong word is chosen,
the rest of the caption builds on that error.

**Beam search** (used in v2) addresses this by maintaining multiple
candidate sequences simultaneously and pruning only the worst ones.
Beam search with width=3 explores 3 parallel captions and returns the best.

**Top-k sampling with temperature** (used in v3) is a probabilistic alternative:
instead of picking the single best token, sample from the top-k tokens
with probabilities scaled by temperature T. This introduces diversity —
the same image can produce slightly different captions each time.

In [ ]:
def caption_generator_greedy(image_feature, model, vocab, max_len=max_len):
    """Generate a caption using greedy decoding.

    Args:
        image_feature: np.array of shape (2048,) — pre-extracted InceptionV3 features
        model: trained Keras captioning model
        vocab: list of vocabulary strings (from vectorizer.get_vocabulary())
        max_len: maximum caption length in tokens

    Returns:
        Generated caption string (without <start>/<end> tokens).
    """
    # Build inverse vocabulary: integer ID → word string
    index_to_word = {idx: word for idx, word in enumerate(vocab)}
    word_to_index = {word: idx for idx, word in enumerate(vocab)}

    # Seed: start token
    in_text = [word_to_index.get('<start>', 2)]

    # Prepare image feature: add batch dimension → (1, 2048)
    img_feat = np.expand_dims(image_feature, axis=0)

    for _ in range(max_len):
        # Pad current sequence to max_len
        seq = pad_sequences([in_text], maxlen=max_len, padding='post', value=0)

        # Forward pass: predict next token probabilities
        # Output shape: (1, vocab_size)
        pred = model.predict([img_feat, seq], verbose=0)

        # Greedy: take the token with highest probability
        next_token_id = np.argmax(pred[0])
        next_word = index_to_word.get(next_token_id, '')

        # Stop at <end> token
        if next_word == '<end>':
            break

        in_text.append(next_token_id)

    # Decode and strip special tokens
    words = [index_to_word.get(i, '') for i in in_text[1:]]  # Skip <start>
    words = [w for w in words if w not in ('<start>', '<end>', '')]  # Clean
    return ' '.join(words)

---

## 10. BLEU Evaluation

### What is BLEU?
**BLEU (Bilingual Evaluation Understudy)** is the standard automatic metric for
evaluating text generation tasks. It measures the overlap between a generated
text and one or more reference texts.

BLEU computes precision for n-grams (sequences of n consecutive words):
- BLEU-1: unigram precision (individual words)
- BLEU-2: bigram precision (pairs of words)
- BLEU-3: trigram precision
- BLEU-4: 4-gram precision

The final **BLEU-4** score is the geometric mean of BLEU-1 through BLEU-4,
plus a **brevity penalty** that penalizes very short captions.

### Why BLEU-4 specifically?
BLEU-4 is the standard for image captioning research (since the original
Show and Tell paper in 2015). It captures 4-gram overlap, which reflects
meaningful phrase-level similarity rather than just word-level overlap.
A score of 0.0 means no 4-gram overlap; 1.0 means perfect match.
State-of-the-art models on COCO score BLEU-4 ~0.35–0.40.

### Limitations of BLEU for image captioning
1. **Requires multiple references:** BLEU was designed for multiple reference translations.
   Instagram data has only 1 caption/image, so any caption that uses different words
   but means the same thing scores 0. Flickr30k/COCO have 5 references, making BLEU
   more meaningful.
2. **Does not capture visual grounding:** BLEU measures text similarity, not whether
   the caption describes the image correctly.
3. **Sensitive to length:** Short generated captions get penalized heavily by brevity penalty.

### Smoothing
For sentences with no n-gram overlap at a particular order (common for short captions),
BLEU-4 would be 0. `SmoothingFunction().method4` (Chen & Cherry, 2014) applies
a small epsilon to all n-gram precisions to avoid this zero-score problem.

In [ ]:
def evaluate_bleu(model, val_keys, image_features_dict, updated_captions_dict,
                  vocab, max_len=max_len, num_samples=100):
    """Compute average BLEU-4 on a sample of validation images.

    Note: Instagram data has only 1 reference caption per image.
    BLEU with a single reference penalizes any synonymous phrasing.
    Results are therefore lower than they would be with 5 references (Flickr30k/COCO).
    """
    smoothie = SmoothingFunction().method4
    bleu_scores = []

    sample_keys = random.sample(val_keys, min(num_samples, len(val_keys)))

    for key in tqdm(sample_keys):
        image_feature = image_features_dict[key]

        # Generate caption
        pred_caption = caption_generator_greedy(image_feature, model, vocab, max_len)

        # Reference: strip <start>/<end>, lowercase, tokenize by space
        ref_caption = updated_captions_dict[key]
        ref_caption = re.sub(r'<start>|<end>', '', ref_caption).strip().lower()
        reference = [ref_caption.split()]

        # Hypothesis: generated caption words
        hypothesis = pred_caption.lower().split()

        score = sentence_bleu(
            reference,
            hypothesis,
            weights=(0.25, 0.25, 0.25, 0.25),  # BLEU-4 equal weights
            smoothing_function=smoothie
        )
        bleu_scores.append(score)

    avg_bleu = sum(bleu_scores) / len(bleu_scores)
    print(f'Average BLEU-4 ({num_samples} samples): {avg_bleu:.4f}')
    return avg_bleu


# ── BLEU evaluation requires the trained model to be loaded.
# ── Run after loading model from checkpoint:
# bleu_v1 = evaluate_bleu(model, val_keys, image_features_dict, updated_captions_dict, vocab)
print('BLEU evaluation: run with trained model loaded.')

---

## 11. Results and Analysis

### Training observations
- Both `model` (Add) and `model1` (Concat) converged to similar validation loss values
- Training was stable with ReduceLROnPlateau and EarlyStopping
- Early stopping typically triggered after ~15–25 epochs

### Inference behavior
Regardless of input image, the model generates phrases like:
- `I love my new`
- `I love my best friend`
- `I love you`
- `link in bio`

This is not random failure — it is a systematic collapse to the mode of the caption distribution.
These phrases appear hundreds of times across the 21k Instagram captions.
The model has learned the prior: 'what is the most common thing Instagram posts say?'
rather than 'what does this image show?'

### Root cause (confirmed in v2 ChatGPT analysis)
1. **1 caption/image:** The model cannot learn image→text alignment with a single noisy signal
2. **Captions not visually grounded:** 'Link in bio!' has zero visual content
3. **Large vocabulary, small dataset:** 10k vocab tokens / 21k images → most tokens appear < 3 times
4. **Loss not masked:** padding tokens contribute to loss (0 tokens dominate long sequences)

**Formal BLEU-4:** Not measured on v1 (no per-image BLEU pipeline implemented at this stage).
The beam-search BLEU measured in v2 on the same data came to **0.026**, confirming near-zero performance.

---

## 12. Key Insights from v1

1. **Architecture is not the bottleneck.** Both `Add` and `Concat` produce the same
   collapsed output. Changing the fusion strategy cannot fix a data problem.

2. **Dataset quality matters more than quantity.** 21k image-caption pairs sounds
   like enough data, but if the captions don't describe the images, it doesn't matter
   how many there are.

3. **Flat 2048-dim features lose spatial information.** InceptionV3 `layers[-2]`
   averages all spatial locations into one vector. The model cannot attend to
   specific regions. v2 fixes this by using the `mixed10` layer: spatial 64×2048.

4. **One caption per image is weak supervision.** Every decision in v3 chooses
   datasets with 5 captions per image (Flickr30k, COCO). The BLEU jump from
   0.026 to 0.134 is largely explained by this single change.

5. **`ReduceLROnPlateau` alone is not enough.** When LR is reduced, training continues
   from the current (potentially overfitted) weights. v2 introduces `ReduceLRReloadBest`
   which reloads the best checkpoint whenever LR is reduced — a significant improvement.

---

## 13. Version Comparison Table

| Version | Encoder | Decoder | Dataset | BLEU-4 | Key Problem |
|---------|---------|---------|---------|--------|-------------|
| **v0 (Colab)** | InceptionV3 `layers[-2]` flat 2048 | LSTM(256) | Instagram | — | Runtime crash: `train_dataset not defined` |
| **v1 (this notebook)** | InceptionV3 `layers[-2]` flat 2048 | LSTM(512), two variants (Add/Concat) | Instagram | ~0.0 | Captions not visually grounded; collapses to 'I love my new' |
| v2 | InceptionV3 `mixed10` spatial (64×2048) | TemporalImageAttention + LSTM(256) | Instagram | 0.026 | Same data problem; attention helps marginally |
| v3 Phase 1 | ConvNeXt-Tiny multi-scale (64,768) | Custom Transformer Decoder | Flickr30k | 0.0674 | Framework switch to PyTorch; better data |
| v3 Phase 2d | ConvNeXt + Perceiver | GPT-2 + CrossAttention (×12) | Flickr30k | 0.0656 | Cross-attention recovers from prepend regression |
| **v3 Phase 3a** | ConvNeXt + Perceiver | GPT-2 + CrossAttention (×12) | **COCO 2017** | **0.1341** | **Best model — COCO's visual captions unlock the jump** |

---

**Next:** `v2_inceptionv3_attention.ipynb` — adds spatial features and a custom attention mechanism
over image regions. Still on Instagram data, but the architectural improvements are significant
and worth documenting as standalone contributions.